# SMART Constrained Probabilistic Variant (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart-constrained_colab.ipynb)

## Objective
- Keep SMART baseline fixed and run constrained-probabilistic variant planning.
- Generate a sweep over temperature/top-k/constraint-weight and select feasible best variant.
- Persist artifacts for thesis-grade reproducibility and benchmark comparison.


In [ ]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-constrained config and validate run context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
SWEEP = dict(cfg.get("sweep", {}))
BASELINE = dict(cfg.get("baseline", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
assert PERSIST_ROOT.strip(), "persist_root is required"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

WOSAC_VARIANT_METRICS_DIR = os.environ.get("WOSAC_VARIANT_METRICS_DIR", "").strip()
print("RUN_TAG:", RUN_TAG)
print("cfg_hash:", cfg_hash)
print("WOSAC_VARIANT_METRICS_DIR:", WOSAC_VARIANT_METRICS_DIR or "<not set>")


In [ ]:
# Step 3: Generate constrained variant grid and selection summary
from src.workflows import run_smart_constrained_flow

flow_bundle = run_smart_constrained_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    baseline_metrics_json=str(BASELINE.get("baseline_metrics_json", "experiments/smart-baseline/results/smart_baseline_v0_metrics.json")),
    variant_metrics_dir=WOSAC_VARIANT_METRICS_DIR,
    max_collision_rate=CONSTRAINTS.get("max_collision_rate"),
    max_offroad_rate=CONSTRAINTS.get("max_offroad_rate"),
    max_tl_violation_rate=CONSTRAINTS.get("max_tl_violation_rate"),
    min_diversity_score=CONSTRAINTS.get("min_diversity_score"),
    temperatures=SWEEP.get("temperatures", [0.8, 1.0, 1.2]),
    top_ks=SWEEP.get("top_ks", [8, 16]),
    constraint_weights=SWEEP.get("constraint_weights", [0.05, 0.1, 0.2]),
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path=str(SMART.get("ckpt_path", "")),
    smart_raw_data_root=str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART.get("install_pyg", True)),
    sync_smart_repo=True,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("selection:")
print(json.dumps(flow_bundle.selection, indent=2, sort_keys=True))
print("num_variants:", len(flow_bundle.variants))
print("first_variant:")
print(json.dumps(flow_bundle.variants[0], indent=2, sort_keys=True))


In [ ]:
# Step 4: Optional command execution for one selected variant
RUN_SETUP = False
RUN_PREPROCESS = False
RUN_TRAIN = False
RUN_VALIDATE = False

variant_map = {v["variant_id"]: v for v in flow_bundle.variants}
default_variant = flow_bundle.selection.get("selected_variant_id") or flow_bundle.variants[0]["variant_id"]
VARIANT_ID_TO_RUN = default_variant
selected = variant_map[VARIANT_ID_TO_RUN]
print("Selected variant:", VARIANT_ID_TO_RUN)

cmds = []
if RUN_SETUP:
    cmds.append(selected["setup_cmd"])
if RUN_PREPROCESS:
    cmds.extend([selected["preprocess_train_cmd"], selected["preprocess_val_cmd"]])
if RUN_TRAIN:
    cmds.append(selected["train_cmd"])
if RUN_VALIDATE:
    cmds.append(selected["validate_cmd"])

for i, cmd in enumerate(cmds, start=1):
    print(f"[exec {i}/{len(cmds)}] {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Execution block done.")


In [ ]:
# Step 5: Write constrained-variant artifact (repo + Drive)
try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = "unknown"

payload = {
    "run_id": "smart_constrained_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "selection": flow_bundle.selection,
    "constraints": flow_bundle.constraints,
    "variants": flow_bundle.variants,
    "baseline": flow_bundle.baseline,
    "notes": [
        "Constrained probabilistic SMART variant artifact.",
        "Set WOSAC_VARIANT_METRICS_DIR to ingest variant metric JSON files named <variant_id>.json.",
    ],
    "provenance": {
        "repo_commit": git_commit,
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_metrics_path = Path("experiments/smart-constrained/results/smart_constrained_v0_metrics.json")
repo_metrics_path.parent.mkdir(parents=True, exist_ok=True)
repo_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_metrics_path = persist_run_dir / "outputs" / "smart_constrained_v0_metrics.json"
drive_metrics_path.parent.mkdir(parents=True, exist_ok=True)
drive_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_metrics_path:", repo_metrics_path)
print("drive_metrics_path:", drive_metrics_path)
